In [0]:
# =============================================================================
# ANÁLISE FINAL - Respondendo as perguntas de negócio
# =============================================================================
# Objetivo: Utilizar a tabela Gold para responder as 6 perguntas definidas
# no objetivo do MVP sobre padrão de compra dos clientes da Olist.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.functions import when

# Leitura da tabela Gold
fato = spark.table("workspace.default.gold_fato_pedidos")

# Filtra apenas pedidos entregues para garantir análises sobre compras concluídas
fato_entregue = fato.filter(F.col("order_status") == "delivered")

print(f"Total de itens entregues para análise: {fato_entregue.count():,}")

# =============================================================================
# PERGUNTA 1: Quais categorias concentram maior volume de pedidos e receita?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 1: Volume de pedidos e receita por categoria")
print("="*60)

p1 = (fato_entregue
    .filter(F.col("categoria_en").isNotNull())
    .groupBy("categoria_en")
    .agg(
        F.count("order_id").alias("volume_itens"),
        F.round(F.sum("preco_item"), 2).alias("receita_total"),
        F.round(F.avg("preco_item"), 2).alias("ticket_medio")
    )
    .orderBy("volume_itens", ascending=False)
)

print("\nTop 10 categorias por volume:")
p1.show(10, truncate=False)

print("\nTop 10 categorias por receita total:")
p1.orderBy("receita_total", ascending=False).show(10, truncate=False)

# =============================================================================
# PERGUNTA 2: Ticket médio por categoria e variação entre regiões
# =============================================================================
# A pergunta tem duas dimensões: ticket médio por categoria e variação
# entre regiões. A análise é dividida em três partes: visão regional,
# visão por categoria e o cruzamento entre ambas.
# =============================================================================

# Mapeamento de estados para as cinco regiões brasileiras
fato_regiao = fato_entregue.withColumn("regiao",
    when(F.col("estado_cliente").isin("SP","RJ","ES","MG"), "Sudeste")
    .when(F.col("estado_cliente").isin("PR","SC","RS"), "Sul")
    .when(F.col("estado_cliente").isin("BA","SE","AL","PE","PB","RN","CE","PI","MA"), "Nordeste")
    .when(F.col("estado_cliente").isin("GO","MT","MS","DF"), "Centro-Oeste")
    .when(F.col("estado_cliente").isin("AM","PA","RO","RR","AC","AP","TO"), "Norte")
    .otherwise("Outros")
)

# --- 2a. Ticket médio por região ---
print("\n" + "="*60)
print("PERGUNTA 2a: Ticket médio por região")
print("="*60)

p2a = (fato_regiao
    .filter(F.col("categoria_en").isNotNull())
    .groupBy("regiao")
    .agg(
        F.round(F.avg("preco_item"), 2).alias("ticket_medio"),
        F.count("order_id").alias("volume_itens"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    .orderBy("ticket_medio", ascending=False)
)

print("\nTicket médio por região:")
p2a.show(truncate=False)

# --- 2b. Ticket médio por categoria ---
print("\n" + "="*60)
print("PERGUNTA 2b: Ticket médio por categoria")
print("="*60)

p2b = (fato_entregue
    .filter(F.col("categoria_en").isNotNull())
    .groupBy("categoria_en")
    .agg(
        F.round(F.avg("preco_item"), 2).alias("ticket_medio"),
        F.count("order_id").alias("volume_itens"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    # Filtra categorias com volume mínimo relevante para evitar que categorias
    # com pouquíssimas vendas distorçam o ranking de ticket médio
    .filter(F.col("volume_itens") >= 100)
    .orderBy("ticket_medio", ascending=False)
)

print("\nTop 10 categorias por ticket médio (mínimo 100 itens vendidos):")
p2b.show(10, truncate=False)

print("\n10 categorias com menor ticket médio:")
p2b.orderBy("ticket_medio", ascending=True).show(10, truncate=False)

# --- 2c. Cruzamento categoria x região ---
print("\n" + "="*60)
print("PERGUNTA 2c: Variação do ticket médio por categoria entre regiões")
print("="*60)

# Seleciona as 5 categorias de maior volume para tornar o cruzamento legível
top_categorias = [row["categoria_en"] for row in (
    fato_entregue
        .filter(F.col("categoria_en").isNotNull())
        .groupBy("categoria_en")
        .count()
        .orderBy("count", ascending=False)
        .limit(5)
        .collect()
)]

print(f"\nCategorias analisadas: {top_categorias}")

# Pivot: categorias nas linhas, regiões nas colunas, ticket médio nos valores
p2c = (fato_regiao
    .filter(F.col("categoria_en").isin(top_categorias))
    .groupBy("categoria_en")
    .pivot("regiao", ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"])
    .agg(F.round(F.avg("preco_item"), 2))
    .orderBy("categoria_en")
)

print("\nTicket médio (R$) por categoria e região:")
p2c.show(truncate=False)

# =============================================================================
# PERGUNTA 3: Quais formas de pagamento são mais usadas e qual o parcelamento médio?
# =============================================================================
# Nota metodológica: forma de pagamento e parcelamento são atributos do
# PEDIDO, não do item. Como a tabela Gold tem granularidade de item, é
# necessário deduplicar por order_id antes de agregar, caso contrário um
# pedido com múltiplos itens seria contado várias vezes.
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 3: Formas de pagamento e parcelamento")
print("="*60)

# Reduz a granularidade de item para pedido, preservando os atributos
# de pagamento que são constantes dentro de um mesmo pedido
pedidos_pagamento = (fato_entregue
    .select("order_id", "forma_pagamento", "parcelas", "valor_total_pedido")
    .dropDuplicates(["order_id"])
)

print(f"\nPedidos únicos entregues: {pedidos_pagamento.count():,}")

p3 = (pedidos_pagamento
    .filter(F.col("forma_pagamento") != "not_defined")
    .groupBy("forma_pagamento")
    .agg(
        F.count("order_id").alias("total_pedidos"),
        F.round(F.avg("parcelas"), 1).alias("media_parcelas"),
        F.round(F.avg("valor_total_pedido"), 2).alias("ticket_medio_pedido")
    )
    .orderBy("total_pedidos", ascending=False)
)

print("\nFormas de pagamento (contagem por pedido único):")
p3.show(truncate=False)

# Percentual de participação de cada forma de pagamento
total_pedidos_validos = (pedidos_pagamento
    .filter(F.col("forma_pagamento") != "not_defined")
    .count())

print(f"\nParticipação percentual sobre {total_pedidos_validos:,} pedidos:")
(p3.withColumn(
    "percentual",
    F.round((F.col("total_pedidos") / total_pedidos_validos) * 100, 1)
).select("forma_pagamento", "total_pedidos", "percentual").show(truncate=False))

# =============================================================================
# PERGUNTA 4: Como o volume de pedidos evoluiu ao longo dos meses?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 4: Evolução mensal do volume de pedidos")
print("="*60)

p4 = (fato_entregue
    .filter(F.col("ano_compra").isin(2017, 2018))
    .groupBy("ano_compra", "mes_compra")
    .agg(
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    .orderBy("ano_compra", "mes_compra")
)

print("\nEvolução mensal (2017-2018):")
p4.show(24, truncate=False)

# =============================================================================
# PERGUNTA 5: Quais estados têm maior concentração de clientes compradores?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 5: Concentração de clientes por estado")
print("="*60)

p5 = (fato_entregue
    .groupBy("estado_cliente")
    .agg(
        F.countDistinct("customer_id").alias("total_clientes"),
        F.countDistinct("order_id").alias("total_pedidos"),
        F.round(F.sum("preco_item"), 2).alias("receita_total")
    )
    .orderBy("total_clientes", ascending=False)
)

print("\nTop 10 estados por clientes:")
p5.show(10, truncate=False)

# =============================================================================
# PERGUNTA 6: Existe relação entre avaliação do pedido e valor gasto?
# =============================================================================
print("\n" + "="*60)
print("PERGUNTA 6: Relação entre avaliação e valor gasto")
print("="*60)

# Consome da camada Silver, mantendo a coerência da arquitetura medalhão.
# O tratamento de valores malformados em review_score já foi aplicado na Silver.
reviews = spark.table("workspace.default.silver_order_reviews")

valor_pedido = (fato_entregue
    .groupBy("order_id")
    .agg(F.round(F.sum("preco_item"), 2).alias("valor_pedido"))
)

p6 = (reviews
    .select("order_id", "review_score")
    .join(valor_pedido, on="order_id", how="inner")
    .groupBy("review_score")
    .agg(
        F.count("order_id").alias("total_pedidos"),
        F.round(F.avg("valor_pedido"), 2).alias("valor_medio_pedido"),
        F.round(F.min("valor_pedido"), 2).alias("valor_minimo"),
        F.round(F.max("valor_pedido"), 2).alias("valor_maximo")
    )
    .orderBy("review_score")
)

print("\nValor médio gasto por nota de avaliação:")
p6.show(truncate=False)

In [0]:
# =============================================================================
# DISCUSSÃO DOS RESULTADOS
# =============================================================================
# Interpretação de negócio das análises realizadas, conectando os números
# obtidos ao problema original definido no objetivo do MVP.
# =============================================================================

print("""
DISCUSSÃO DOS RESULTADOS
========================

PERGUNTA 1: Categorias por volume e receita
-------------------------------------------
Os rankings de volume e de receita divergem de forma reveladora.
Cama, mesa e banho lidera em volume com 10.953 itens, mas cai para
terceiro lugar em receita. Relogios e presentes faz o movimento
oposto: setimo em volume, segundo em receita, sustentado pelo ticket
medio de R$ 199,04, o maior entre as categorias de alto volume.
A categoria cool_stuff sequer aparece no top 10 de volume, mas figura
em oitavo em receita.

A implicacao pratica e que uma estrategia focada apenas em volume de
transacoes subestimaria categorias de alto valor unitario. Para um
marketplace que cobra comissao sobre valor transacionado, categorias
como relogios e presentes sao desproporcionalmente relevantes em
relacao ao seu volume.

PERGUNTA 2: Ticket medio por categoria e entre regioes
-------------------------------------------------------
A analise foi conduzida em tres dimensoes.

Por regiao, Norte (R$ 159,19) e Nordeste (R$ 148,51) apresentam
ticket medio superior ao Sudeste (R$ 114,36), apesar de concentrarem
volume muito menor.

Por categoria, a amplitude e enorme: de R$ 55,55 em alimentos e
bebidas a R$ 1.098,92 em computadores, uma razao de vinte vezes.
Categorias de baixo ticket tendem a ser de consumo recorrente,
enquanto as de alto ticket concentram bens duraveis.

O cruzamento entre as duas dimensoes e o achado mais relevante deste
MVP. A hipotese inicial era que as regioes Norte e Nordeste comprassem
categorias diferentes, mais caras. O pivot desmente isso: dentro da
mesma categoria, o ticket medio no Norte e sistematicamente superior
ao do Sudeste. Em saude e beleza a diferenca chega a 65% (R$ 198,49
contra R$ 120,46). O padrao se repete nas cinco categorias analisadas,
sem excecao.

Isso significa que a diferenca regional nao e efeito de composicao de
mix de produtos, mas de comportamento de compra. Duas explicacoes sao
plausiveis: consumidores em regioes com menor oferta de varejo fisico
recorrem ao e-commerce para compras de maior valor, que justificam o
frete e o prazo mais longos; ou compram em maior quantidade por
transacao para diluir o custo logistico. Ambas apontam para a mesma
conclusao de negocio: o custo de aquisicao de cliente nessas regioes
pode ser compensado por um valor por transacao significativamente maior.

PERGUNTA 3: Formas de pagamento
--------------------------------
O cartao de credito responde por 75% das transacoes com parcelamento
medio de 3,6 vezes, confirmando o comportamento caracteristico do
consumidor brasileiro de fracionar compras mesmo em valores moderados.
Considerando o ticket medio de R$ 182,67, a parcela tipica fica em
torno de R$ 50.

O dado mais contraintuitivo e o boleto. A expectativa seria que fosse
usado predominantemente em compras de baixo valor, ja que nao permite
parcelamento. No entanto, seu ticket medio de R$ 176,33 e praticamente
equivalente ao do cartao. Isso sugere que a escolha pelo boleto nao e
determinada pelo valor da compra, mas por perfil de consumidor,
provavelmente clientes sem acesso a credito ou que preferem evitar o
uso do cartao. Restringir ou desincentivar o boleto teria impacto
direto em receita, nao apenas em transacoes de baixo valor.

O cartao de debito representa apenas 1,5% das transacoes, refletindo
o periodo analisado, anterior a popularizacao de meios de pagamento
instantaneo no Brasil.

PERGUNTA 4: Evolucao mensal
----------------------------
O ano de 2017 mostra crescimento acelerado e consistente, com o volume
mensal multiplicando-se por quase dez vezes entre janeiro (750 pedidos)
e novembro (7.289). O pico de novembro e quase certamente efeito de
Black Friday, e a queda em dezembro para 5.513 sugere antecipacao de
compras de fim de ano.

O comportamento em 2018 e qualitativamente diferente. O volume se
estabiliza entre 6.000 e 7.000 pedidos mensais, sem tendencia clara de
crescimento. Dois pontos merecem atencao: o patamar de janeiro de 2018
(7.069) e superior ao de dezembro de 2017, indicando que o crescimento
de novembro incorporou base de clientes de forma permanente e nao foi
apenas sazonal. E a estabilizacao pode indicar tanto maturidade de
mercado quanto limitacao de capacidade operacional, hipotese que
exigiria dados adicionais para ser testada.

PERGUNTA 5: Concentracao geografica
-------------------------------------
Sao Paulo concentra 40.501 clientes, mais do que Rio de Janeiro e
Minas Gerais somados. Os tres estados do Sudeste representam cerca de
58% da base, refletindo a concentracao economica brasileira, mas
tambem possivelmente a concentracao de vendedores da plataforma, que
reduz custo e prazo de frete para compradores proximos.

Limitacao metodologica importante: o numero de clientes e identico ao
de pedidos em todos os estados. Isso ocorre porque no dataset da Olist
o campo customer_id e gerado por pedido, nao por pessoa. A identificacao
de cliente recorrente exigiria o campo customer_unique_id, disponivel
na tabela original mas nao incorporado a esta modelagem. Portanto esta
analise mede concentracao de transacoes por estado, nao de individuos
distintos.

PERGUNTA 6: Avaliacao e valor gasto
-------------------------------------
Existe relacao, mas nao e linear. Pedidos com nota 1 apresentam o maior
valor medio de todos (R$ 164,87), enquanto as notas 2 a 5 se concentram
numa faixa estreita entre R$ 127 e R$ 143. O padrao nao e de correlacao
continua, e de concentracao de insatisfacao severa nas compras de
maior valor.

A explicacao mais provavel envolve assimetria de expectativa e de risco.
Compras caras geram expectativa proporcionalmente maior, e qualquer
falha de entrega ou divergencia de produto tem consequencia mais grave
para o comprador. Alem disso, itens de maior valor tendem a ser maiores
fisicamente, o que aumenta a probabilidade de avaria no transporte e
de atraso logistico.

O valor maximo na nota 1 (R$ 13.440,00) e o maior de toda a base,
superando inclusive o maximo entre pedidos com nota 5, o que reforca
essa leitura.

A distribuicao de volume e tranquilizadora: 56.664 pedidos com nota 5
contra 9.292 com nota 1, proporcao de seis para um. A experiencia
majoritaria e positiva. O problema e especifico e enderecavel,
concentrado no segmento de alto ticket.

CONCLUSAO GERAL
---------------
O pipeline revelou um marketplace em transicao entre fase de
crescimento acelerado (2017) e consolidacao (2018), com tres padroes
estruturais claros.

O primeiro e a dissociacao entre volume e valor na dimensao geografica.
O Sudeste concentra 68% dos itens e 70% da receita, mas apresenta o
menor ticket medio do pais. Norte e Nordeste invertem a relacao, e o
cruzamento por categoria confirmou que isso e comportamento de compra,
nao mix de produtos.

O segundo e a centralidade do credito parcelado. Tres quartos das
transacoes passam por cartao de credito, mas o boleto sustenta ticket
medio equivalente, indicando que o meio de pagamento reflete perfil de
acesso a credito e nao valor da compra.

O terceiro e a concentracao de insatisfacao no alto ticket. Pedidos com
avaliacao minima tem valor medio 22% superior a media geral, apontando
uma lacuna especifica de experiencia em compras de maior valor.

Do ponto de vista de Engenharia de Dados, o exercicio reforcou que a
qualidade do pipeline se define na camada mais proxima da origem. O
problema de leitura multilinha nos reviews passaria despercebido em uma
inspecao superficial e teria contaminado silenciosamente todas as
analises subsequentes. Foi a estrutura em camadas que permitiu corrigir
na origem e reprocessar sem retrabalho.
""")